### Clone Repository

In [ ]:
!git clone https://github.com/qaz812345/TrackNetV3.git
%cd TrackNetV3

Cloning into 'TrackNetV3'...
remote: Enumerating objects: 240, done.
remote: Counting objects: 100% (131/131), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 240 (delta 117), reused 105 (delta 105), pack-reused 109 (from 1)
Receiving objects: 100% (240/240), 2.81 MiB | 11.86 MiB/s, done.
Resolving deltas: 100% (134/134), done.
/content/TrackNetV3


###  Install project requirements

In [ ]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 81.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Install extra libraries

In [ ]:
!pip install parse tqdm opencv-python

### Check project directory and folders

In [ ]:
import os
print("Current Directory:", os.getcwd())
print("Files in current directory:", os.listdir())
if os.path.exists('ckpts'):
    print("Files in ckpts folder:", os.listdir('ckpts'))
else:
    print("The 'ckpts' folder does not exist!")

Current Directory: /content/TrackNetV3
Files in current directory: ['figure', 'LICENSE', 'preprocess.py', 'correct_label.py', 'corrected_test_label', 'dataset.py', 'train.py', 'error_analysis.py', '__pycache__', 'test.py', 'model.py', '.git', 'README.md', 'utils', 'predict.py', 'generate_mask_data.py', 'requirements.txt', '.gitignore']
The 'ckpts' folder does not exist!


###  Run Prediction

### Video 1

In [ ]:
# Move to the project directory
%cd /content/TrackNetV3

!python predict.py \
--video_file /content/1_13_12.mp4 \
--tracknet_file "/content/drive/MyDrive/TrackNetV3_ckpts/ckpts/TrackNet_best.pt" \
--inpaintnet_file "/content/drive/MyDrive/TrackNetV3_ckpts/ckpts/InpaintNet_best.pt" \
--save_dir /content/TrackNetV3/outputs \
--output_video \
--large_video \
--batch_size 1

/content/TrackNetV3
Generate median image...
Median image generated.
Video length: 655
649it [03:50,  2.82it/s]
100% 641/641 [00:02<00:00, 261.58it/s]
OpenCV: FFMPEG: tag 0x34504d46/'FMP4' is not supported with codec id 12 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x7634706d/'mp4v'
Done.


### Display predicted video

In [ ]:
from IPython.display import HTML
from base64 import b64encode

video_path = '/content/TrackNetV3/outputs/1_13_12_predict.mp4'

mp4 = open(video_path,'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(f"""
<video width=600 controls>
      <source src="{data_url}" type="video/mp4">
</video>
""")

### Video 2 (Fix video codec)

In [ ]:
%cd /content/TrackNetV3

# Deep Patch: Force standard VideoWriter settings
predict_file = 'predict.py'
with open(predict_file, 'r') as f:
    lines = f.readlines()

new_lines = []
for line in lines:
    # Force the codec to MJPG or mp4v (widely supported)
    if "cv2.VideoWriter_fourcc" in line:
        new_lines.append("    fourcc = cv2.VideoWriter_fourcc(*'MJPG')\n")
    # Force the output extension to .avi if the encoder is being picky
    elif "save_path = " in line and "mp4" in line:
        new_lines.append(line.replace(".mp4", ".avi"))
    else:
        new_lines.append(line)

with open(predict_file, 'w') as f:
    f.writelines(new_lines)

print("✅ Deep patch applied: Forced MJPG codec and AVI compatibility.")

# Define paths
VIDEO_INPUT = "/content/2022-08-30_18-38-37_dataset_set1_039_004426_004475_A_13.mp4"
TRACKNET_WEIGHTS = "/content/drive/MyDrive/TrackNetV3_ckpts/ckpts/TrackNet_best.pt"
INPAINT_WEIGHTS = "/content/drive/MyDrive/TrackNetV3_ckpts/ckpts/InpaintNet_best.pt"

# Run Inference
!python predict.py \
--video_file "{VIDEO_INPUT}" \
--tracknet_file "{TRACKNET_WEIGHTS}" \
--inpaintnet_file "{INPAINT_WEIGHTS}" \
--save_dir /content/TrackNetV3/outputs \
--output_video \
--large_video \
--batch_size 1

# Check for the output
print("\n--- Checking Outputs ---")
!ls /content/TrackNetV3/outputs/

/content/TrackNetV3
✅ Deep patch applied: Forced MJPG codec and AVI compatibility.
Generate median image...
Median image generated.
Video length: 49
43it [00:15,  2.78it/s]
100% 35/35 [00:00<00:00, 136.82it/s]
OpenCV: FFMPEG: tag 0x34363268/'h264' is not supported with codec id 27 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x31637661/'avc1'
[h264_v4l2m2m @ 0x27f8ff40] Could not find a valid device
[h264_v4l2m2m @ 0x27f8ff40] can't configure encoder
[ERROR:0@20.785] global cap_ffmpeg_impl.hpp:3514 open Could not open codec h264_v4l2m2m, error: Unspecified error (-22)
[ERROR:0@20.785] global cap_ffmpeg_impl.hpp:3531 open VIDEOIO/FFMPEG: Failed to initialize VideoWriter
Done.

--- Checking Outputs ---
1_13_12_ball.csv
1_13_12.mp4
2022-08-30_18-38-37_dataset_set1_039_004426_004475_A_13_ball.csv
shuttle_tracking_ball.csv
shuttle_tracking.mp4


### Draw tracking dots on video using csv

In [ ]:
import cv2
import pandas as pd
import numpy as np
from tqdm import tqdm

# Paths
video_in = "/content/2022-08-30_18-38-37_dataset_set1_039_004426_004475_A_13.mp4"
csv_in = "/content/TrackNetV3/outputs/2022-08-30_18-38-37_dataset_set1_039_004426_004475_A_13_ball.csv"
video_out = "/content/TrackNetV3/outputs/FINAL_TRACKED_VIDEO.mp4"

# Load Data
df = pd.read_csv(csv_in)
cap = cv2.VideoCapture(video_in)
fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Force Standard 'mp4v' Codec
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(video_out, fourcc, fps, (w, h))

print("Creating tracked video from CSV data...")
for i in tqdm(range(len(df))):
    ret, frame = cap.read()
    if not ret:
        break

    # Get coordinates from CSV
    vis = df.loc[i, 'Visibility']
    x = int(df.loc[i, 'X'])
    y = int(df.loc[i, 'Y'])

    # If ball is visible, draw a yellow circle and trajectory
    if vis > 0:
        cv2.circle(frame, (x, y), 5, (0, 255, 255), -1)
        # Draw a small tail (last 3 frames)
        for j in range(max(0, i-3), i):
            prev_x, prev_y = int(df.loc[j, 'X']), int(df.loc[j, 'Y'])
            if df.loc[j, 'Visibility'] > 0:
                cv2.line(frame, (prev_x, prev_y), (x, y), (0, 255, 0), 2)

    out.write(frame)

cap.release()
out.release()
print(f"\nDone! Your video is saved at: {video_out}")